# Predictive Maintenance Model — Spindle Failure Probability
**Catalog:** `manufacturing_facility_ops`  
**Experiment:** `predictive_maintenance_spindle`  
**Description:** Uses MLflow catalog and scores all machines.

## 0. Setup

In [1]:
import mlflow
import mlflow.sklearn
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from mlflow.tracking import MlflowClient
from pyspark.sql import functions as F

# NO autolog — this is what was causing the long runtime
mlflow.set_experiment("predictive_maintenance_spindle")

FEATURES = [
    "spindle_incidents_6m",
    "total_incidents",
    "total_downtime_hrs",
    "days_since_last_maint",
    "total_maintenance_cost_usd"
]

print("Setup complete — autolog disabled for fast demo run")
print(f"Features: {FEATURES}")

Setup complete — autolog disabled for fast demo run
Features: ["spindle_incidents_6m", "total_incidents", "total_downtime_hrs", "days_since_last_maint", "total_maintenance_cost_usd"]


## 1. Training Data

In [2]:
# Read real machine data from silver — cast in Spark, collect as plain Python
rows = spark.table("manufacturing_facility_ops.silver.machine_status") \
    .selectExpr(
        "CAST(spindle_incidents AS DOUBLE)          AS spindle_incidents_6m",
        "CAST(total_incidents AS DOUBLE)            AS total_incidents",
        "CAST(total_downtime_hrs AS DOUBLE)         AS total_downtime_hrs",
        "CAST(days_since_last_maint AS DOUBLE)      AS days_since_last_maint",
        "CAST(total_maintenance_cost_usd AS DOUBLE) AS total_maintenance_cost_usd",
        "CASE WHEN status IN ('Under Review','Maintenance') THEN 1.0 ELSE 0.0 END AS spindle_failure"
    ).collect()

real_X = np.array([[float(r[f] or 0) for f in FEATURES] for r in rows], dtype=np.float64)
real_y = np.array([float(r["spindle_failure"]) for r in rows], dtype=np.float64)

np.random.seed(42)

# Small synthetic dataset — 100 healthy, 50 at-risk
healthy_X = np.column_stack([
    np.random.choice([0.0, 1.0], 100, p=[0.85, 0.15]),
    np.random.randint(0, 4, 100).astype(np.float64),
    np.random.uniform(0, 12, 100),
    np.random.randint(10, 90, 100).astype(np.float64),
    np.random.uniform(200, 1500, 100),
])
healthy_y = np.zeros(100, dtype=np.float64)

atrisk_X = np.column_stack([
    np.random.choice([2.0, 3.0, 4.0], 50, p=[0.5, 0.3, 0.2]),
    np.random.randint(3, 10, 50).astype(np.float64),
    np.random.uniform(15, 50, 50),
    np.random.randint(60, 200, 50).astype(np.float64),
    np.random.uniform(2000, 8000, 50),
])
atrisk_y = np.ones(50, dtype=np.float64)

X_all = np.vstack([real_X, healthy_X, atrisk_X])
y_all = np.concatenate([real_y, healthy_y, atrisk_y])

idx   = np.random.permutation(len(X_all))
X_all = X_all[idx]
y_all = y_all[idx]

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")
print(f"Failure rate     : {y_all.mean():.1%}")

Training samples : 126
Test samples     : 32
Failure rate     : 32.9%


## 2. TrainModel and Log to MLflow

In [3]:
with mlflow.start_run(run_name="spindle_failure_demo_v1"):

    # Small fast model — sufficient for demo
    params = {"n_estimators": 50, "max_depth": 5, "max_features": 3}
    rf = RandomForestClassifier(**params, random_state=42)
    rf.fit(X_train, y_train)

    y_pred  = rf.predict(X_test)
    y_proba = rf.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    # Log manually — no autolog
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", round(acc, 3))
    mlflow.log_metric("f1_score", round(f1,  3))
    mlflow.log_metric("roc_auc",  round(auc, 3))
    mlflow.log_param("features",  str(FEATURES))
    mlflow.sklearn.log_model(rf, "spindle_failure_model")

    best_run_id = mlflow.active_run().info.run_id
    best_model  = rf

print(f"✅  Model trained")
print(f"    accuracy = {acc:.3f} | f1 = {f1:.3f} | auc = {auc:.3f}")
print(f"    run_id   = {best_run_id}")

✅  Model trained
    accuracy = 0.969 | f1 = 0.952 | auc = 0.955
    run_id   = 7a123a1a196d4a44a182b474764e638c


## 3. Register Model to MLflow Catalog

In [4]:
model_uri  = f"runs:/{best_run_id}/spindle_failure_model"
model_name = "manufacturing_facility_ops.gold.spindle_failure_predictor"

registered = mlflow.register_model(model_uri=model_uri, name=model_name)

client = MlflowClient()
client.set_registered_model_tag(model_name, "use_case",    "Predictive Maintenance")
client.set_registered_model_tag(model_name, "asset",       "CNC Spindle — M-102")
client.set_registered_model_tag(model_name, "facility",    "CE-Facility-01")
client.set_registered_model_tag(model_name, "agent",       "Maintenance Copilot")
client.set_registered_model_tag(model_name, "features",    str(FEATURES))

print(f"✅  Model registered: {model_name}")
print(f"    Version : {registered.version}")
print(f"    Status  : {registered.status}")

✅  Model registered: manufacturing_facility_ops.gold.spindle_failure_predictor
    Version : 5
    Status  : READY


## 4. Score All Machines from Gold Layer

In [6]:
# Cast in Spark, collect as plain Python — no toPandas, no Arrow issues
score_rows = spark.table("manufacturing_facility_ops.gold.machine_risk_profile") \
    .selectExpr(
        "machine_id",
        "machine_name",
        "production_line",
        "status",
        "risk_level",
        "CAST(spindle_incidents_6m AS DOUBLE)      AS spindle_incidents_6m",
        "CAST(total_incidents AS DOUBLE)            AS total_incidents",
        "CAST(total_downtime_hrs AS DOUBLE)         AS total_downtime_hrs",
        "0.0                                        AS days_since_last_maint",
        "CAST(total_maintenance_cost_usd AS DOUBLE) AS total_maintenance_cost_usd"
    ).collect()

X_live = np.array([
    [
        float(r["spindle_incidents_6m"]       or 0),
        float(r["total_incidents"]            or 0),
        float(r["total_downtime_hrs"]         or 0),
        float(r["days_since_last_maint"]      or 0),
        float(r["total_maintenance_cost_usd"] or 0),
    ]
    for r in score_rows
], dtype=np.float64)

probs = best_model.predict_proba(X_live)[:, 1].round(2)
pcts  = (probs * 100).round(1)
recs  = [
    "IMMEDIATE MAINTENANCE" if p >= 0.75
    else "SCHEDULE MAINTENANCE" if p >= 0.50
    else "MONITOR"              if p >= 0.25
    else "NO ACTION"
    for p in probs
]

print("--- MACHINE FAILURE PROBABILITY SCORES ---")
print(f"{'machine_id':<10} {'name':<18} {'line':<10} {'status':<14} {'spindle':<9} {'fail_%':<8} {'recommendation'}")
print("-" * 90)
for i, r in enumerate(score_rows):
    print(
        f"{r['machine_id']:<10} "
        f"{r['machine_name']:<18} "
        f"{r['production_line']:<10} "
        f"{r['status']:<14} "
        f"{int(r['spindle_incidents_6m'] or 0):<9} "
        f"{pcts[i]:<8} "
        f"{recs[i]}"
    )

--- MACHINE FAILURE PROBABILITY SCORES ---
machine_id name               line       status         spindle   fail_%   recommendation
------------------------------------------------------------------------------------------
M-101      CNC Lathe 101      Line-1     Operational    0         0.0      NO ACTION
M-102      CNC Lathe 102      Line-3     Under Review   3         100.0    IMMEDIATE MAINTENANCE
M-103      CNC Mill 103       Line-2     Operational    0         0.0      NO ACTION
M-104      CNC Mill 104       Line-4     Operational    0         0.0      NO ACTION
M-105      CNC Lathe 105      Line-5     Operational    0         0.0      NO ACTION
M-106      CNC Lathe 106      Line-5     Operational    0         0.0      NO ACTION
M-107      Drill Press 107    Line-6     Operational    0         0.0      NO ACTION
M-108      CNC Mill 108       Line-2     Maintenance    0         0.0      NO ACTION


## 5. Save Scores to Gold Layer

In [7]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

score_data = [
    (
        score_rows[i]["machine_id"],
        score_rows[i]["machine_name"],
        score_rows[i]["production_line"],
        score_rows[i]["status"],
        score_rows[i]["risk_level"],
        float(probs[i]),
        float(pcts[i]),
        recs[i]
    )
    for i in range(len(score_rows))
]

schema = StructType([
    StructField("machine_id",              StringType(), True),
    StructField("machine_name",            StringType(), True),
    StructField("production_line",         StringType(), True),
    StructField("status",                  StringType(), True),
    StructField("risk_level",              StringType(), True),
    StructField("failure_probability",     DoubleType(), True),
    StructField("failure_probability_pct", DoubleType(), True),
    StructField("model_recommendation",    StringType(), True),
])

spark.createDataFrame(score_data, schema) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("manufacturing_facility_ops.gold.machine_failure_scores")

print("✅  manufacturing_facility_ops.gold.machine_failure_scores saved")
print("\n--- M-102 FINAL SCORE ---")
spark.table("manufacturing_facility_ops.gold.machine_failure_scores") \
    .filter("machine_id = 'M-102'") \
    .select("machine_id", "status", "risk_level", "failure_probability_pct", "model_recommendation") \
    .show(truncate=False)

✅  manufacturing_facility_ops.gold.machine_failure_scores saved

--- M-102 FINAL SCORE ---


+----------+------------+----------+-----------------------+---------------------+
|machine_id|status      |risk_level|failure_probability_pct|model_recommendation |
+----------+------------+----------+-----------------------+---------------------+
|M-102     |Under Review|CRITICAL  |100.0                  |IMMEDIATE MAINTENANCE|
+----------+------------+----------+-----------------------+---------------------+

